# Phase 1: Exploratory Data Analysis

**Dataset:** Give Me Some Credit (Kaggle)  
**Goal:** Understand the data shape, target imbalance, missing values, distributions, and outliers before any modelling.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_raw

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load Dataset

In [ ]:
df = load_raw()
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

In [ ]:
df.dtypes

## 2. Target Distribution

Credit datasets are heavily imbalanced — defaults are rare events. This directly affects which models and metrics we use.

In [ ]:
counts = df['serious_dlqin2yrs'].value_counts().sort_index()
pcts   = df['serious_dlqin2yrs'].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['No Default (0)', 'Default (1)'], counts.values,
              color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.5)
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'{pct:.1f}%', ha='center', fontweight='bold')
ax.set_title('Target Distribution', fontsize=13)
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

print(f"Default rate: {pcts[1]:.2f}%  →  roughly 1 in {int(100 / pcts[1])} borrowers defaults")

## 3. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
mv = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
mv[mv['missing_count'] > 0]

## 4. Descriptive Statistics

In [ ]:
df.describe().T.round(4)

## 5. Feature Distributions

Clipped at the 99th percentile so extreme outliers don't compress the histogram.

In [ ]:
cols = df.columns.tolist()
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cols):
    upper = df[col].quantile(0.99)
    df[col].clip(upper=upper).hist(ax=axes[i], bins=40,
                                   color='#4C72B0', edgecolor='white')
    axes[i].set_title(col, fontsize=8)
    axes[i].tick_params(labelsize=7)

for j in range(len(cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions (clipped at 99th pct)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 6. Correlation Matrix

In [ ]:
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Outlier Detection

In [ ]:
checks = {
    'utilization > 1 (maxed out)':        (df['revolving_utilization_of_unsecured_lines'] > 1).sum(),
    'utilization > 10 (likely error)':     (df['revolving_utilization_of_unsecured_lines'] > 10).sum(),
    'debt_ratio > 1':                      (df['debt_ratio'] > 1).sum(),
    'age < 18':                            (df['age'] < 18).sum(),
    'monthly_income > 500k':               (df['monthly_income'] > 500_000).sum(),
    'times_90_days_late > 90 (impossible)':(df['number_of_times90_days_late'] > 90).sum(),
}
for label, count in checks.items():
    print(f"  {label:48s}: {count:>6,}")

## 8. Default Rate by Feature Decile

Binning each feature into deciles and plotting the default rate per bin reveals monotonic relationships — useful for feature selection and understanding risk drivers.

In [ ]:
key_features = [
    'revolving_utilization_of_unsecured_lines',
    'age',
    'debt_ratio',
    'monthly_income',
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(key_features):
    clipped = df[col].clip(upper=df[col].quantile(0.99))
    bins = pd.qcut(clipped, q=10, duplicates='drop')
    rate = df.groupby(bins, observed=True)['serious_dlqin2yrs'].mean() * 100
    rate.plot(kind='bar', ax=axes[i], color='#DD8452', edgecolor='white')
    axes[i].set_title(f'Default Rate by {col}', fontsize=9)
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)

plt.suptitle('Default Rate by Feature Decile', fontsize=13)
plt.tight_layout()
plt.show()

## Key Findings

| Finding | Implication |
|---|---|
| ~7% default rate | Use stratified splits; accuracy is misleading — use AUC/KS/Gini |
| `monthly_income` has ~20% missing | Impute with median (Phase 2) |
| Utilization values > 10 exist | Cap outliers before modelling |
| `number_of_times90_days_late` most correlated with target | Will be top SHAP feature |
| Age negatively correlated with default | Older borrowers are lower risk |